# Ranking models with `scorio.rank`

We use [Scorio](https://github.com/mohsenhariri/scorio) for scoring, ranking and
aggregating stochastic model responses. This notebook covers `scorio.rank`, which takes
a response tensor for several models and returns their ranking.

Every method takes the same first argument, a tensor `R` of shape `L x M x N`, models by
questions by trials. One Scorio Trace split is exactly that, 20 models by 30 questions by
80 seeds. Methods return ranks by default, and `(ranks, scores)` with `return_scores=True`.

|  |  |
| --- | --- |
| module | [`scorio/rank`](https://github.com/mohsenhariri/scorio/tree/main/scorio/rank) |
| method reference | [`scorio/rank/README.md`](https://github.com/mohsenhariri/scorio/blob/main/scorio/rank/README.md) |
| paper | [Ranking Reasoning LLMs under Test-Time Scaling](https://aclanthology.org/2026.acl-long.1544/), ACL 2026 ([arXiv](https://arxiv.org/abs/2603.10960)) |
| video | [walkthrough](https://github.com/user-attachments/assets/b5bc4ca1-a62b-412f-b6ec-ac7eb58481c4) |
| docs | [scorio.readthedocs.io/en/latest/api/rank](https://scorio.readthedocs.io/en/latest/api/rank.html) |
| install | `pip install scorio` |

The data comes from the [Scorio Trace](https://huggingface.co/datasets/harimo/scorio-trace) dataset, see [trace.ipynb](https://github.com/mohsenhariri/scorio/blob/main/notebooks/datasets/trace/trace.ipynb).

In [1]:
import pandas as pd
from datasets import load_dataset
from scorio import rank

repo_name = "harimo/scorio-trace"
task = "aime25"

rows = (load_dataset(repo_name, "meta", split=task)
        .select_columns(["model_key", "data_id", "seed", "is_correct"])
        .to_pandas()
        .sort_values(["model_key", "data_id", "seed"]))

models = sorted(rows.model_key.unique())
R = rows.is_correct.to_numpy().astype(int).reshape(len(models), 30, 80)

print(R.shape, "models x questions x seeds")

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

(20, 30, 80) models x questions x seeds


## Bayes@N ranking

`rank.bayes` orders models by their [Bayes@N](https://arxiv.org/abs/2510.04265) scores.
Rank 1 is the best model. Asking for the scores as well shows what the ranking was built from.

In [2]:
ranks, scores = rank.bayes(R, return_scores=True)

leaderboard = pd.DataFrame({"rank": ranks.astype(int), "Bayes@N": scores.round(3)}, index=models)
display(leaderboard.sort_values("rank"))

,rank,Bayes@N
Qwen3-30B-A3B-Thinking-2507,1,0.798
Qwen3-4B-Thinking-2507,2,0.726
gpt-oss-20b_high,3,0.695
gpt-oss-20b_medium,4,0.686
Phi-4-reasoning-plus,5,0.679
AceReason-Nemotron-1.1-7B,6,0.648
Phi-4-reasoning,7,0.598
gpt-oss-20b_low,8,0.596
OpenThinker2-32B,9,0.593
Light-R1-14B-DS,10,0.587


## The method matters

The same tensor, seven ranking methods. `bayes`, `rasch` and `thompson` return the
same order here. Borda and pairwise win rate move Phi-4-reasoning from 7th to 10th,
because they count how often a model beats another model question by question rather than
how many questions it solves.

Elo is the outlier, and that is expected. It is a sequential rating system that walks
through matches one at a time, so on a complete design like this one it reflects the order
of updates as much as the outcomes. Use it when your data arrives as a match stream, not
as a full tensor.

In [3]:
methods = ["bayes", "borda", "win_rate", "bradley_terry", "elo", "rasch", "thompson"]

comparison = pd.DataFrame({m: getattr(rank, m)(R).astype(int) for m in methods}, index=models)
comparison = comparison.sort_values("bayes")

display(comparison)
print("models ranked differently by at least two methods:",
      int((comparison.nunique(axis=1) > 1).sum()), "of", len(models))

,bayes,borda,win_rate,bradley_terry,elo,rasch,thompson
Qwen3-30B-A3B-Thinking-2507,1,1,1,1,12,1,1
Qwen3-4B-Thinking-2507,2,2,2,2,3,2,2
gpt-oss-20b_high,3,4,4,3,13,3,3
gpt-oss-20b_medium,4,5,6,4,4,4,4
Phi-4-reasoning-plus,5,3,3,5,2,5,5
AceReason-Nemotron-1.1-7B,6,6,5,6,9,6,6
Phi-4-reasoning,7,10,10,7,16,7,7
gpt-oss-20b_low,8,8,8,9,8,8,8
OpenThinker2-32B,9,7,7,8,6,9,9
Light-R1-14B-DS,10,11,11,10,1,10,10


models ranked differently by at least two methods: 20 of 20


## The budget matters more

Same models, same questions, same method, only the number of samples per question changes.
At one sample per question gpt-oss-20b_medium leads and Qwen3-30B is second. By 80 samples
Qwen3-30B is first and gpt-oss-20b_medium is fourth.

Most of the table moves. This is the point of the ranking paper. A leaderboard is a
statement about a sampling budget, and reporting it without one hides most of the story.

In [4]:
budgets = [1, 2, 4, 8, 16, 32, 80]

sweep = pd.DataFrame({f"n={n}": rank.bayes(R[:, :, :n]).astype(int) for n in budgets}, index=models)
sweep = sweep.sort_values("n=80")

display(sweep)
print("models whose rank changes between n=1 and n=80:",
      int((sweep["n=1"] != sweep["n=80"]).sum()), "of", len(models))

,n=1,n=2,n=4,n=8,n=16,n=32,n=80
Qwen3-30B-A3B-Thinking-2507,2,1,1,1,1,1,1
Qwen3-4B-Thinking-2507,8,6,4,3,2,2,2
gpt-oss-20b_high,4,2,3,2,3,3,3
gpt-oss-20b_medium,1,4,2,4,4,4,4
Phi-4-reasoning-plus,6,8,7,5,5,5,5
AceReason-Nemotron-1.1-7B,6,6,7,7,6,7,6
Phi-4-reasoning,4,4,6,8,8,8,7
gpt-oss-20b_low,2,3,4,6,7,6,8
OpenThinker2-32B,9,11,12,10,10,10,9
Light-R1-14B-DS,9,9,9,9,9,9,10


models whose rank changes between n=1 and n=80: 18 of 20


## Other APIs

Around 40 methods in seven families, all with the same signature. Voting
([`voting.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/rank/voting.py)) has
Copeland, Schulze, ranked pairs, Kemeny-Young and majority judgment. Paired comparison
([`bradley_terry.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/rank/bradley_terry.py))
has Bradley-Terry with Davidson and Rao-Kupper tie models plus MAP variants. Item response
theory ([`irt.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/rank/irt.py)) has
Rasch, 2PL, 3PL, marginal maximum likelihood and multidimensional IRT. There are also
graph methods, seriation, Hodge rank and the Luce listwise family.

The full table with a reference for each method is in
[`scorio/rank/README.md`](https://github.com/mohsenhariri/scorio/blob/main/scorio/rank/README.md).